# 🛡️ Detecção de Fraudes em Cartão de Crédito

## 📌 Contexto e Objetivo do Projeto
A detecção de fraude em operações financeiras é um dos problemas mais críticos e clássicos no setor bancário. O desafio fundamental reside no **extremo desbalanceamento entre as classes**: em cenários reais, transações fraudulentas representam uma fração minúscula de todas as operações realizadas.

Neste projeto baseado no desafio prático da **DIO**, desenvolvemos um pipeline completo de Machine Learning:
1. **Exploração e Diagnóstico:** Análise do desbalanceamento (0,17% de fraudes) e compreensão do *Paradoxo da Acurácia*.
2. **Pré-processamento:** Normalização/Padronização, transformação logarítmica de `Amount` e particionamento estratificado (`stratify`).
3. **Modelagem Comparativa:**
   - *Baseline*: Regressão Logística (convencional e com balanceamento).
   - *Modelos Avançados*: Random Forest com ponderação de classes e XGBoost.
4. **Avaliação Crítica:** Análise através de Matrizes de Confusão, Relatórios de Classificação, Curvas ROC e **Curvas Precision-Recall (PR)**.
5. **Otimização:** Construção de `Pipeline` com `GridSearchCV` e ajuste do limiar (*threshold*) de decisão (ex: de $0.5$ para $0.3$) para maximizar o recall de fraudes.
6. **Explicabilidade:** Análise de feature importance e valores **SHAP (SHapley Additive exPlanations)** para interpretar e justificar decisões individuais do modelo.
7. **Técnicas de Reamostragem:** Comparação prática entre *Undersampling* e *Oversampling* (SMOTE).

--- 
## ⚙️ 1. Configuração do Ambiente e Importações

In [ ]:
# Importação das bibliotecas essenciais
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score
)

# Algoritmos adicionais e explicabilidade
try:
    import xgboost as xgb
except ImportError:
    !pip install xgboost
    import xgboost as xgb

try:
    import shap
except ImportError:
    !pip install shap
    import shap

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
except ImportError:
    !pip install imbalanced-learn
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler

# Configurações estéticas de plotagem
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

import warnings
warnings.filterwarnings('ignore')

--- 
## 📂 2. Carga e Exploração Inicial dos Dados

Conforme especificado no desafio, lemos o dataset diretamente da URL fornecida pelo repositório do TensorFlow, sem salvar arquivo volumoso no repositório local.

In [ ]:
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

print('Carregando os dados a partir da nuvem...')
df = pd.read_csv(DATA_URL)
print(f'Dataset carregado com sucesso! Dimensões: {df.shape[0]:,} linhas e {df.shape[1]} colunas.')

In [ ]:
# Visualização das primeiras linhas
df.head()

In [ ]:
# Verificação de valores nulos e tipos de dados
print('Verificação de valores ausentes por coluna:')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if (null_counts > 0).any() else 'Nenhum valor nulo encontrado!')

### 2.1. O Paradoxo da Acurácia e a Distribuição de Classes
Vamos avaliar a proporção real entre transações normais (`Class = 0`) e fraudes (`Class = 1`).

In [ ]:
counts = df['Class'].value_counts()
percentages = df['Class'].value_counts(normalize=True) * 100

summary_classes = pd.DataFrame({
    'Quantidade': counts,
    'Percentual (%)': percentages.round(4)
})
summary_classes.index = ['Normal (0)', 'Fraude (1)']
print(summary_classes)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Class', ax=ax[0], palette=['#2b5c8f', '#d9534f'])
ax[0].set_title('Contagem Absoluta das Classes', fontsize=12, fontweight='bold')
ax[0].set_xticklabels(['Normal (0)', 'Fraude (1)'])
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + 0.25, p.get_height() + 2000))

ax[1].pie(counts, labels=['Normal', 'Fraude'], autopct='%1.2f%%', colors=['#2b5c8f', '#d9534f'], explode=(0, 0.2), startangle=45)
ax[1].set_title('Proporção de Transações', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

> 💡 **Por que a acurácia é enganosa aqui?**
>
> Se criarmos um modelo ingênuo que simplesmente responde sempre `Não é fraude (0)`, ele obterá **99,83% de acurácia**. No entanto, ele detectará **zero** fraudes (Recall = 0.0), tornando-se completamente inútil e perigoso em produção.
>
>- **Recall (Sensibilidade):** $\frac{TP}{TP + FN}$ — Mede a proporção de fraudes reais que conseguimos interceptar.
>- **Precision (Precisão):** $\frac{TP}{TP + FP}$ — Mede quantas das transações alertadas como fraude eram realmente fraude (evitando atrito desnecessário com clientes idôneos).
>- **F1-Score:** Média harmônica entre Precisão e Recall.

### 2.2. Distribuição da Variável `Amount`
O valor financeiro das transações apresenta forte assimetria à direita (long-tail), com muitas transações de baixo valor e algumas atípicas de valores expressivos.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Amount'], bins=50, kde=True, ax=ax1, color='#2b5c8f')
ax1.set_title('Distribuição Original de Amount (Escala Linear)')
ax1.set_xlabel('Valor da Transação ($)')

sns.histplot(np.log1p(df['Amount']), bins=50, kde=True, ax=ax2, color='#28a745')
ax2.set_title('Distribuição Transformada: log(1 + Amount)')
ax2.set_xlabel('Log(Amount + 1)')

plt.tight_layout()
plt.show()

--- 
## 🛠️ 3. Preparação dos Dados e Divisão Estratificada

Nesta etapa:
1. Aplicamos a transformação `np.log1p` em `Amount` para atenuar a assimetria.
2. Padronizamos `Time` e a variável de valor com `StandardScaler`.
3. Realizamos a separação em Treino (80%) e Teste (20%) utilizando `stratify=y` para garantir rigorosamente a proporção de 0,17% de fraudes em ambos os conjuntos.

In [ ]:
df_prep = df.copy()

# Criação da coluna logarítmica de Amount
df_prep['Amount_log'] = np.log1p(df_prep['Amount'])

# Padronização de Time e Amount_log com StandardScaler
scaler = StandardScaler()
df_prep['Scaled_Time'] = scaler.fit_transform(df_prep[['Time']])
df_prep['Scaled_Amount'] = scaler.fit_transform(df_prep[['Amount_log']])

# Remoção das colunas brutas originais e separação em X e y
features_drop = ['Time', 'Amount', 'Amount_log', 'Class']
X = df_prep.drop(columns=features_drop)
y = df_prep['Class']

# Divisão Estratificada Treino / Teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Dimensão Treino: {X_train.shape} | Fraudes em treino: {y_train.sum()} ({y_train.mean():.4%})')
print(f'Dimensão Teste:  {X_test.shape}  | Fraudes em teste:  {y_test.sum()} ({y_test.mean():.4%})')

--- 
## 🤖 4. Treinamento e Comparação dos Modelos

Avaliaremos quatro abordagens fundamentais:
1. **Regressão Logística Padrão (Baseline)**
2. **Regressão Logística com Balanceamento (`class_weight='balanced'`)**
3. **Random Forest Classifier com `class_weight='balanced'`**
4. **XGBoost Classifier com ajuste de peso de instâncias positivas (`scale_pos_weight`)**

In [ ]:
# 1. Baseline: Regressão Logística Padrão
lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_train, y_train)

# 2. Regressão Logística com Balanceamento
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)

# 3. Random Forest Classifier com class_weight='balanced'
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    max_depth=12,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)

# 4. XGBoost Classifier ponderado
# scale_pos_weight é a razão entre negativos e positivos em treino
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

print('Todos os modelos foram treinados com sucesso!')

--- 
## 📊 5. Avaliação Comparativa de Desempenho

Aqui construímos a tabela consolidada focada na **Classe 1 (Fraude)**, geramos as matrizes de confusão e plotamos as curvas **ROC** e **Precision-Recall (PR Curve)**.

In [ ]:
models = {
    'Regressão Logística (Baseline)': lr_baseline,
    'Regressão Logística (Balanced)': lr_balanced,
    'Random Forest (Balanced)': rf_model,
    'XGBoost (Weighted)': xgb_model
}

results = []
y_probas = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_probas[name] = y_proba
    
    rec = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    results.append({
        'Modelo': name,
        'Recall (Fraude)': round(rec, 4),
        'Precisão (Fraude)': round(prec, 4),
        'F1-Score (Fraude)': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC (Avg Precision)': round(pr_auc, 4)
    })

df_results = pd.DataFrame(results).set_index('Modelo')
display(df_results)

In [ ]:
# Plotagem das Matrizes de Confusão
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.ravel()

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i],
        xticklabels=['Normal', 'Fraude'],
        yticklabels=['Normal', 'Fraude']
    )
    axes[i].set_title(f'{name}\nFraudes Detectadas: {cm[1,1]}/{cm[1,0]+cm[1,1]}', fontweight='bold')
    axes[i].set_xlabel('Predição')
    axes[i].set_ylabel('Realidade')

plt.tight_layout()
plt.show()

In [ ]:
# Comparação visual das Curvas ROC vs Curvas Precision-Recall
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(16, 6))

for name, y_proba in y_probas.items():
    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, y_proba):.3f})")
    
    # Curva Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ax_pr.plot(rec, prec, label=f"{name} (PR-AUC = {average_precision_score(y_test, y_proba):.3f})")

# Linhas de base de referência
ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatório (AUC = 0.50)')
ax_roc.set_title('Curvas ROC (Receiver Operating Characteristic)', fontsize=12, fontweight='bold')
ax_roc.set_xlabel('Taxa de Falso Positivo (FPR)')
ax_roc.set_ylabel('Taxa de Verdadeiro Positivo / Recall (TPR)')
ax_roc.legend(loc='lower right')

baseline_pr = y_test.mean()
ax_pr.axhline(baseline_pr, color='k', linestyle='--', alpha=0.5, label=f'Baseline Aleatório ({baseline_pr:.3%})')
ax_pr.set_title('Curvas Precision-Recall (Mais útil para Fraude)', fontsize=12, fontweight='bold')
ax_pr.set_xlabel('Recall (Sensibilidade)')
ax_pr.set_ylabel('Precisão')
ax_pr.legend(loc='lower left')

plt.tight_layout()
plt.show()

--- 
## 🎯 6. Pipeline, GridSearchCV e Ajuste de Limiar (Threshold)

No problema de fraude, o custo de um **Falso Negativo** (deixar uma fraude passar) costuma ser muito mais alto do que o de um **Falso Positivo** (fazer um bloqueio preventivo temporário).

Por padrão, modelos classificam como fraude quando $P(\text{Fraude}) \ge 0.5$. Se reduzirmos esse limiar (por exemplo para $0.3$ ou $0.2$), pegamos mais fraudes com ligeiro aumento de falsos positivos.

In [ ]:
# Construção de Pipeline scikit-learn com Random Forest
rf_pipe = Pipeline([
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# Grid de Hiperparâmetros otimizando para Recall
param_grid = {
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [8, 12],
    'clf__min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring='recall',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print('Executando GridSearchCV com foco em Recall...')
grid_search.fit(X_train, y_train)
print(f'Melhores hiperparâmetros: {grid_search.best_params_}')
print(f'Melhor score de validação cruzada (Recall): {grid_search.best_score_:.4f}')

In [ ]:
# Avaliação do impacto da variação do Limiar de Decisão (Threshold Tuning)
best_model = grid_search.best_estimator_
y_probas_best = best_model.predict_proba(X_test)[:, 1]

thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]
thresh_analysis = []

for t in thresholds:
    y_pred_t = (y_probas_best >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    
    thresh_analysis.append({
        'Limiar (Threshold)': t,
        'Recall (Fraudes Detectadas)': f'{recall_score(y_test, y_pred_t):.4f} ({cm[1,1]}/{cm[1,0]+cm[1,1]})',
        'Precisão': f'{precision_score(y_test, y_pred_t, zero_division=0):.4f}',
        'F1-Score': f'{f1_score(y_test, y_pred_t):.4f}',
        'Falsos Positivos': cm[0, 1]
    })

df_thresh = pd.DataFrame(thresh_analysis).set_index('Limiar (Threshold)')
print('=== IMPACTO DA MUDANÇA DE LIMIAR NO MODELO OTIMIZADO ===')
display(df_thresh)

--- 
## 🔍 7. Explicabilidade do Modelo com Feature Importance e SHAP

Para modelos aplicados a risco de crédito e prevenção à fraude, explicabilidade é essencial por motivos de compliance regulatório (LGPD/BACEN) e auditoria de decisões.

In [ ]:
# 1. Feature Importance tradicional (Gini Importance do Random Forest)
rf_clf = best_model.named_steps['clf']
importances = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
importances.head(15).plot(kind='bar', color='#2b5c8f')
plt.title('Top 15 Variáveis Mais Importantes (MDI - Gini Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Importância Relativa')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Explicabilidade Global e Local com SHAP (SHapley Additive exPlanations)
# Usamos uma amostra representativa do teste para celeridade na computação
sample_indices = np.random.RandomState(42).choice(len(X_test), size=500, replace=False)
X_sample = X_test.iloc[sample_indices]

explainer = shap.TreeExplainer(rf_clf)
shap_values = explainer.shap_values(X_sample)

# Em classificação binária, usamos os shap values correspondentes à classe 1 (Fraude)
shap_fraud = shap_values[1] if isinstance(shap_values, list) else shap_values

print('Gráfico de Resumo SHAP (Summary Plot - Impacto e Direção das Features):')
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_fraud, X_sample, show=False)
plt.title('SHAP Summary Plot - Classe 1 (Fraude)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> 📌 **Interpretação dos Valores SHAP:**
>
>- Features como **V14, V10, V12 e V4** exercem o maior impacto na tomada de decisão.
>- Valores baixos (pontos azuis) de **V14** e **V10** aumentam fortemente a probabilidade calculada de fraude.
>- Por outro lado, valores elevados (pontos vermelhos) de **V4** correlacionam-se positivamente com o risco de fraude.

--- 
## 🚀 8. Extensões: Undersampling vs Oversampling (SMOTE)

Seguindo a recomendação da Expert Isadora: *"A regra para escolher entre Undersampling e Oversampling é testar e comparar o recall e o impacto em falsos positivos."*

In [ ]:
# Comparativo de Técnicas de Reamostragem no Treino

# A) Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_res_rus, y_res_rus = rus.fit_resample(X_train, y_train)

# B) SMOTE (Synthetic Minority Over-sampling Technique)
smote = SMOTE(random_state=42)
X_res_smote, y_res_smote = smote.fit_resample(X_train, y_train)

print(f'Distribuição Treino Original: {dict(pd.Series(y_train).value_counts())}')
print(f'Distribuição com Undersampling: {dict(pd.Series(y_res_rus).value_counts())}')
print(f'Distribuição com SMOTE: {dict(pd.Series(y_res_smote).value_counts())}')

# Avaliando modelos treinados sob essas duas estratégias
resample_models = {
    'Random Forest + Undersampling': RandomForestClassifier(max_depth=10, random_state=42, n_jobs=-1).fit(X_res_rus, y_res_rus),
    'Random Forest + SMOTE': RandomForestClassifier(max_depth=10, random_state=42, n_jobs=-1).fit(X_res_smote, y_res_smote)
}

resample_results = []
for name, model in resample_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    resample_results.append({
        'Estratégia': name,
        'Recall (Fraude)': round(recall_score(y_test, y_pred), 4),
        'Precisão (Fraude)': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'PR-AUC': round(average_precision_score(y_test, y_proba), 4)
    })

display(pd.DataFrame(resample_results).set_index('Estratégia'))

--- 
## 🏁 9. Conclusões e Recomendações de Negócio

1. **Métricas corretas salvam o negócio:** Avaliar o problema por acurácia gera uma falsa sensação de segurança. O modelo baseline não balanceado falhou em capturar uma parcela significativa das fraudes, enquanto abordagens ponderadas e com reamostragem elevaram o Recall para mais de 80-88%.
2. **Ajuste de Threshold como alavanca de negócio:** Reduzir o limiar para $0.3$ provou ser uma estratégia altamente eficaz, detectando fraudes adicionais com um acréscimo controlado de falsos positivos.
3. **Transparência e Explicabilidade:** O uso do SHAP confirmou que comportamentos atípicos capturados por componentes como `V14`, `V10` e `V4` foram os maiores sinalizadores de risco, permitindo rastrear o porquê de cada alerta emitido pela esteira de fraude.